### Determining The Optimal Number of hidden layers and neurons for an ANN
---

- Generally obtained by experimentation but can be archieved faster by keeping some points in mind.
    1. **Start Simple:** Start with simple architecture, increase complexity as it needs.
    2. **Grid / Random Search:** Use to try different different architecture.
    3. **Cross Validation:** Evaluates performance of different architecture.
    4. **Heuristics and Rules of Thumb:** Some heuristic and empirical rules can provide starting points. like:
        - | Neurons in HL | should be between the size of input and output layer.
        - Common practice is to start with 1 or 2 HL.

> Importing Libraries
---

In [5]:
%pip install scikeras

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------------ --------- 1.8/2.4 MB 12.6 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/560.5 kB ? eta -:--:--
   ---------------------------------------- 560.5/560.5 kB 9.6 MB/s  0:00:00

   ----------------- ---------------------- 3/7 [markdown-it-py]
   ----------------- ---------------------- 3/7 [markdown-it-py]
   ---------------------- ----------------- 4/7 [rich]
   ---------------------- ----------------- 4/7 [rich]
   ---------------------- ----------------- 4/7 [rich]
   ---------------------- ----------------- 4/7 [rich]
  Attempting uninstall: keras
   ---------------------- ----------------- 4/7 [rich]
    Found existing installation: keras 2.15.0
   ---------------------- ----------------- 4/7 [rich]
   ---------------------------- ----------- 5/7 [keras]
   ---------------------------- -------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires keras<2.16,>=2.15.0, but you have keras 3.15.1 which is incompatible.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping 
import pickle

> Dataset Work
---

In [2]:
data = pd.read_csv('assets/Churn_Modelling.csv')

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

> Saved Models
---

In [ ]:
with open('models/ann-customer-churn-prediction-label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('models/ann-customer-churn-prediction-onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('models/ann-customer-churn-prediction-scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

> Defining a generic function to create model and try different different parameters(KerasClassifier)
---

In [4]:
def create_ann_model(neuron=32, layers=1):
    model = Sequential()
    model.add(Dense(neuron, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neuron, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model


> Created a Keras Classifier
---

In [8]:
model = KerasClassifier(neuron=32, layers=1, build_fn=create_ann_model, verbose=1)

> Defined grid search parmater
---

In [9]:
param_grid = {
    'neuron': [16, 32, 64, 128],
    'layers': [1,2],
    # 'batch_size': [10, 20],
    'epochs': [50, 100]
}

> Performing Grid Search
---

In [10]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, y_train)

d:\CODE\04-Projects\ann-customer-churn-prediction\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\CODE\04-Projects\ann-customer-churn-prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8025 - loss: 0.4614
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8216 - loss: 0.4158
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8357 - loss: 0.3898
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8514 - loss: 0.3610
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8599 - loss: 0.3480
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8602 - loss: 0.3408
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8612 - loss: 0.3390
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8624 - loss: 0.3340
Epoch 9/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8635 - loss: 0.3328
Epoch 10/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8645 - loss: 0.3308
Epoch 11/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8664 - loss: 0.3291
Epoch 12/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

In [11]:
print(f'Best: {grid_result.best_score_} and using: {grid_result.best_params_}')

Best: 0.8561247454720823 and using: {'epochs': 50, 'layers': 2, 'neuron': 32}
